In [1]:
# Setup
!pip install openai

from openai import AzureOpenAI
import os


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
!pip install python-dotenv
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
endpoint = os.getenv("OPENAI_ENDPOINT")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
# Azure Setup
azure_endpoint = os.getenv("AZURE_ENDPOINT")
client = AzureOpenAI(
    api_key=api_key,
    api_version="2024-02-15-preview",
    azure_endpoint=azure_endpoint
)

DEPLOYMENT_NAME = "gpt-4o-mini"  # your deployed model name

In [4]:
# Hotel Knowledge Base
hotel_docs = [
    "Check-in time is 3 PM and check-out time is 11 AM.",
    "The hotel restaurant is open from 7 AM to 10 PM.",
    "Guests can cancel reservations up to 24 hours before check-in for a full refund.",
    "Free Wi-Fi is available throughout the hotel.",
    "Pets are not allowed except for service animals."
]

In [5]:
# Input Guardrail 
def input_guardrail(user_query):
    blocked_terms = ["hack", "illegal", "steal"]
    
    for term in blocked_terms:
        if term in user_query.lower():
            return False, "Sorry, I cannot help with that request."
    
    return True, user_query

In [6]:
# Simple “RAG Retrieval”
def retrieve_context(query):
    # SUPER simple similarity simulation
    relevant_docs = [doc for doc in hotel_docs if any(word in doc.lower() for word in query.lower().split())]
    
    # fallback if nothing matched
    return relevant_docs[:2] if relevant_docs else hotel_docs[:2]

In [7]:
# Prompt Orchestration
def build_prompt(user_query):
    context = retrieve_context(user_query)
    context_text = "\n".join(context)
    
    system_prompt = """
    You are a helpful hotel assistant.
    Answer clearly and politely.
    Do not provide medical or unsafe advice.
    """
    
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Context:\n{context_text}\n\nQuestion: {user_query}"}
    ]

In [8]:
# LLM API Call
def call_llm(messages):
    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=messages,
        temperature=0.3
    )
    
    return response.choices[0].message.content

In [9]:
# Output Guardrail
def output_guardrail(response):
    banned_phrases = ["dosage", "prescription", "medical advice"]
    
    for phrase in banned_phrases:
        if phrase in response.lower():
            return "I cannot provide medical advice. Please consult a professional."
    
    return response

In [10]:
# Full Pipeline
def hotel_assistant(user_query):
    
    # Input guardrail
    is_valid, result = input_guardrail(user_query)
    if not is_valid:
        return result
    
    # Prompt orchestration (includes RAG)
    messages = build_prompt(user_query)
    
    # LLM API call
    raw_response = call_llm(messages)
    print("'raw response: " + raw_response + "'")
    
    # Output guardrail
    final_response = "final response: " + output_guardrail(raw_response)
    
    return final_response

Example Runs

In [11]:
# ✅ Normal Query (RAG + Prompt)
hotel_assistant("What time is check-in?")

'raw response: Check-in time is at 3 PM.'


'final response: Check-in time is at 3 PM.'

In [12]:
# ✅ Context-based query
hotel_assistant("Can I cancel my booking?")

'raw response: To assist you with your cancellation, please check the specific cancellation policy associated with your booking. Policies can vary based on the rate you chose or the terms of your reservation. If you need further assistance, feel free to provide more details about your booking, and I can guide you on the next steps.'


'final response: To assist you with your cancellation, please check the specific cancellation policy associated with your booking. Policies can vary based on the rate you chose or the terms of your reservation. If you need further assistance, feel free to provide more details about your booking, and I can guide you on the next steps.'

In [13]:
# ✅ Input Guardrail Example
hotel_assistant("How can I hack the hotel system?")

'Sorry, I cannot help with that request.'

In [14]:
# ✅ Output Guardrail Example
hotel_assistant("What medicine should I take if I feel sick?")

'raw response: I'm sorry to hear that you're not feeling well. However, I can't provide medical advice or recommend specific medications. I suggest consulting with a healthcare professional or a pharmacist for the best guidance. If you need assistance with anything else, feel free to ask!'


'final response: I cannot provide medical advice. Please consult a professional.'